In [44]:
import refinitiv.data as rd
import pandas as pd
import numpy as np

rd.open_session()

<refinitiv.data.session.Definition object at 0x11380ddc0 {name='workspace'}>

In [45]:
universe = pd.read_csv("../../data/stoxx600_universe.csv")
STOCK = universe["RIC"].sample(10, random_state=99).tolist()
print("Osakkeet:", STOCK)

PARAMS = {"SDate": "2024-01-01", "EDate": "2025-12-31", "Frq": "D", "Curn": "EUR"}

Osakkeet: ['SRENH.S', 'AKRBP.OL', 'TLIT.MI', 'AAL.L', 'ABVX.PA', 'SVT.L', 'FER.MC', 'RAAG.DE', 'EQTAB.ST', 'ANTO.L']


In [46]:
df = rd.get_data(
    universe=STOCK,
    fields=[
        "TR.PriceClose.date",
        "TR.PriceClose",
        "TR.CompanyMarketCap",
        "TR.SharesOutstanding",
        "TR.PriceToCFPerShare",       # REF_PCF
        "TR.F.NetCashFlowOp",         # OpCF (total)
        "TR.CFPSActValue(Period=LTM)",            # CFPS Actual
        "TR.F.NetCFOpPerShr(Period=LTM)",         # OpCF per share
    ],
    parameters=PARAMS
)

print(f"Sarakkeet ({len(df.columns)}):", df.columns.tolist())
print(f"{len(df)} riviä, {df['Instrument'].nunique()} osaketta")
df.head()

Sarakkeet (9): ['Instrument', 'Date', 'Price Close', 'Company Market Cap', 'Outstanding Shares', 'Price To Cash Flow Per Share (Daily Time Series Ratio)', 'Net Cash Flow from Operating Activities', 'Cash Flow Per Share - Actual', 'Cash Flow from Operations per Share']
5054 riviä, 10 osaketta


/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


,Instrument,Date,Price Close,Company Market Cap,Outstanding Shares,Price To Cash Flow Per Share (Daily Time Series Ratio),Net Cash Flow from Operating Activities,Cash Flow Per Share - Actual,Cash Flow from Operations per Share
0,SRENH.S,2024-01-03,102.589553,32571906664.969501,290405370,14.766154,2784590864.68493,<NA>,7.247412
1,SRENH.S,2024-01-04,103.14453,32748110291.578201,290405370,14.875927,2784590864.68493,<NA>,7.247412
2,SRENH.S,2024-01-05,103.272911,32788871037.308102,290405370,14.891835,2784590864.68493,<NA>,7.247412
3,SRENH.S,2024-01-08,104.334407,33125893184.8125,290405370,15.055774,2784590864.68493,<NA>,7.247412
4,SRENH.S,2024-01-09,103.699341,32924261252.375999,290405370,14.939596,2784590864.68493,<NA>,7.247412


In [47]:
# Tyyppimuunnokset
df.iloc[:, 1] = pd.to_datetime(df.iloc[:, 1])
for c in df.columns[2:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Refinitivin sarake-nimet sellaisenaan — lyhennetään vain viittauksia varten
REF_PCF = df["Price To Cash Flow Per Share (Daily Time Series Ratio)"]
Price   = df["Price Close"]
MktCap  = df["Company Market Cap"]
Shares  = df.groupby("Instrument")["Outstanding Shares"].ffill()
OpCF    = df.groupby("Instrument")["Net Cash Flow from Operating Activities"].ffill()
CFPS    = df.groupby("Instrument")["Cash Flow Per Share - Actual"].ffill()
OpCFPS  = df.groupby("Instrument")["Cash Flow from Operations per Share"].ffill()

# Itselasketut P/CF — vain ne jotka täsmäsivät (#joo)
df["CALC_MktCap_div_OpCF"]      = MktCap / OpCF
df["CALC_Price_div_OpCF_Shares"] = Price / (OpCF / Shares)
df["CALC_Price_div_CFPS_Act"]    = Price / CFPS
df["CALC_Price_div_OpCFPS"]      = Price / OpCFPS

print(f"{len(df)} riviä")
df.head()

5054 riviä


,Instrument,Date,Price Close,Company Market Cap,Outstanding Shares,Price To Cash Flow Per Share (Daily Time Series Ratio),Net Cash Flow from Operating Activities,Cash Flow Per Share - Actual,Cash Flow from Operations per Share,CALC_MktCap_div_OpCF,CALC_Price_div_OpCF_Shares,CALC_Price_div_CFPS_Act,CALC_Price_div_OpCFPS
0,SRENH.S,2024-01-03,102.589553,32571906664.969501,290405370,14.766154,2784590864.68493,<NA>,7.247412,11.697197,10.699079,<NA>,14.155335
1,SRENH.S,2024-01-04,103.14453,32748110291.578201,290405370,14.875927,2784590864.68493,<NA>,7.247412,11.760475,10.756957,<NA>,14.231911
2,SRENH.S,2024-01-05,103.272911,32788871037.308102,290405370,14.891835,2784590864.68493,<NA>,7.247412,11.775113,10.770346,<NA>,14.249625
3,SRENH.S,2024-01-08,104.334407,33125893184.8125,290405370,15.055774,2784590864.68493,<NA>,7.247412,11.896144,10.88105,<NA>,14.396091
4,SRENH.S,2024-01-09,103.699341,32924261252.375999,290405370,14.939596,2784590864.68493,<NA>,7.247412,11.823734,10.814819,<NA>,14.308464


In [48]:
# Haetaan samat kentät LTM-periodilla
ltm = rd.get_data(
    universe=STOCK,
    fields=[
        "TR.F.NetCashFlowOp.date",
        "TR.F.NetCashFlowOp",
        "TR.CFPSActValue",
        "TR.F.NetCFOpPerShr",
    ],
    parameters={**PARAMS, "Period": "LTM", "SDate": "2023-01-01"}
)

print(f"LTM: {len(ltm)} riviä, {ltm['Instrument'].nunique()} osaketta")
print(f"Sarakkeet: {ltm.columns.tolist()}")

# Kattavuus per osake
for stock in STOCK:
    sub = ltm[ltm.Instrument == stock]
    n = sub.iloc[:, 2:].apply(pd.to_numeric, errors="coerce").notna().any(axis=1).sum()
    print(f"  {stock}: {n} rivit joissa dataa")

ltm.head(10)

LTM: 7579 riviä, 10 osaketta
Sarakkeet: ['Instrument', 'Date', 'Net Cash Flow from Operating Activities', 'Cash Flow Per Share - Actual', 'Cash Flow from Operations per Share']
  SRENH.S: 750 rivit joissa dataa
  AKRBP.OL: 751 rivit joissa dataa
  TLIT.MI: 759 rivit joissa dataa
  AAL.L: 758 rivit joissa dataa
  ABVX.PA: 766 rivit joissa dataa
  SVT.L: 758 rivit joissa dataa
  FER.MC: 724 rivit joissa dataa
  RAAG.DE: 762 rivit joissa dataa
  EQTAB.ST: 751 rivit joissa dataa
  ANTO.L: 758 rivit joissa dataa


/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to

,Instrument,Date,Net Cash Flow from Operating Activities,Cash Flow Per Share - Actual,Cash Flow from Operations per Share
0,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
1,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
2,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
3,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
4,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
5,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
6,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
7,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
8,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874
9,SRENH.S,2022-06-30,3636745920.14112,<NA>,12.222874


In [49]:
# Yhdistä LTM-data päivädataan
ltm.columns = ["Instrument", "Date", "OpCF_LTM", "CFPS_LTM", "OpCFPS_LTM"]
ltm["Date"] = pd.to_datetime(ltm["Date"]).dt.normalize()
for c in ["OpCF_LTM", "CFPS_LTM", "OpCFPS_LTM"]:
    ltm[c] = pd.to_numeric(ltm[c], errors="coerce")

df = df.sort_values(["Instrument", "Date"]).dropna(subset=[df.columns[1]])
ltm = ltm.sort_values(["Instrument", "Date"]).dropna(subset=["Date"]).drop_duplicates(subset=["Instrument", "Date"])

parts = []
for stock in STOCK:
    d = df[df.Instrument == stock].copy().reset_index(drop=True)
    l = ltm[ltm.Instrument == stock][["Date", "OpCF_LTM", "CFPS_LTM", "OpCFPS_LTM"]].copy().reset_index(drop=True)
    if len(l) > 0:
        merged = pd.merge_asof(d, l, on="Date", direction="backward")
    else:
        merged = d
        for c in ["OpCF_LTM", "CFPS_LTM", "OpCFPS_LTM"]:
            merged[c] = np.nan
    parts.append(merged)

df = pd.concat(parts, ignore_index=True)

# LTM-pohjaiset P/CF
Price = df["Price Close"]
MktCap = df["Company Market Cap"]
df["CALC_MktCap_div_OpCF_LTM"]   = MktCap / df["OpCF_LTM"]
df["CALC_Price_div_CFPS_LTM"]    = Price / df["CFPS_LTM"]
df["CALC_Price_div_OpCFPS_LTM"]  = Price / df["OpCFPS_LTM"]

# Vertailu kaikki kaavat
ref_col = "Price To Cash Flow Per Share (Daily Time Series Ratio)"
REF_PCF = df[ref_col]
all_cols = ["CALC_MktCap_div_OpCF", "CALC_Price_div_OpCF_Shares",
            "CALC_Price_div_CFPS_Act", "CALC_Price_div_OpCFPS",
            "CALC_MktCap_div_OpCF_LTM", "CALC_Price_div_CFPS_LTM", "CALC_Price_div_OpCFPS_LTM"]

print("=== Kokonaistaso: ei-LTM vs LTM ===")
for col in all_cols:
    mask = REF_PCF.notna() & df[col].notna()
    if mask.sum() > 0:
        diff = ((df.loc[mask, col] - REF_PCF[mask]) / REF_PCF[mask] * 100).abs()
        print(f"  {col}: median {diff.median():.1f}%, mean {diff.mean():.1f}%")

print("\n=== Per osake (median |ero %|) ===")
for stock in STOCK:
    sub = df[df.Instrument == stock]
    ref = sub[ref_col]
    print(f"\n{stock}:")
    for col in all_cols:
        mask = ref.notna() & sub[col].notna()
        if mask.sum() > 0:
            diff = ((sub.loc[mask, col] - ref[mask]) / ref[mask] * 100).abs().median()
            print(f"    {col}: {diff:.1f}%")
        else:
            print(f"    {col}: ei dataa")

=== Kokonaistaso: ei-LTM vs LTM ===
  CALC_MktCap_div_OpCF: median 14.9%, mean 20.5%
  CALC_Price_div_OpCF_Shares: median 16.5%, mean 23.3%
  CALC_Price_div_CFPS_Act: median 6.1%, mean 8.7%
  CALC_Price_div_OpCFPS: median 7.1%, mean 15.9%
  CALC_MktCap_div_OpCF_LTM: median 12.5%, mean 24.0%
  CALC_Price_div_CFPS_LTM: median 6.4%, mean 7.2%
  CALC_Price_div_OpCFPS_LTM: median 8.6%, mean 21.1%

=== Per osake (median |ero %|) ===

SRENH.S:
    CALC_MktCap_div_OpCF: 3.5%
    CALC_Price_div_OpCF_Shares: 9.7%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 4.7%
    CALC_MktCap_div_OpCF_LTM: 4.8%
    CALC_Price_div_CFPS_LTM: ei dataa
    CALC_Price_div_OpCFPS_LTM: 5.8%

AKRBP.OL:
    CALC_MktCap_div_OpCF: 8.9%
    CALC_Price_div_OpCF_Shares: 9.0%
    CALC_Price_div_CFPS_Act: 3.7%
    CALC_Price_div_OpCFPS: 3.5%
    CALC_MktCap_div_OpCF_LTM: 5.7%
    CALC_Price_div_CFPS_LTM: 5.1%
    CALC_Price_div_OpCFPS_LTM: 5.0%

TLIT.MI:
    CALC_MktCap_div_OpCF: 41.2%
    CALC_Price_div_O

In [50]:
# Kolme vaihtoehtoa rinnakkain
out = df[["Instrument", "Date"]].copy()
out.columns = ["Instrument", "Date"]

# 1) Refinitivin oma P/CF (käännetty CF/P)
ref_col = "Price To Cash Flow Per Share (Daily Time Series Ratio)"
out["REF_PCF"] = df[ref_col]
out["REF_CFP"] = 1 / df[ref_col]

# 2) Price / CFPS_LTM
out["PCF_CFPS_LTM"] = df["CALC_Price_div_CFPS_LTM"]
out["CFP_CFPS_LTM"] = 1 / df["CALC_Price_div_CFPS_LTM"]

# 3) OpCF / MktCap (suoraan CF/P)
out["CFP_OpCF_MktCap"] = df["OpCF_LTM"] / df["Company Market Cap"]
out["PCF_OpCF_MktCap"] = df["Company Market Cap"] / df["OpCF_LTM"]

out.to_csv("pcf_3_vaihtoehtoa.csv", index=False)
print(f"pcf_3_vaihtoehtoa.csv ({len(out)} riviä, {out.Instrument.nunique()} osaketta)")

# NaN %
print("\nNaN %:")
for c in out.columns[2:]:
    pct = out[c].isna().mean() * 100
    print(f"  {c}: {pct:.1f}%")

out

pcf_3_vaihtoehtoa.csv (5054 riviä, 10 osaketta)

NaN %:
  REF_PCF: 12.3%
  REF_CFP: 12.3%
  PCF_CFPS_LTM: 85.1%
  CFP_CFPS_LTM: 85.1%
  CFP_OpCF_MktCap: 0.0%
  PCF_OpCF_MktCap: 0.0%


,Instrument,Date,REF_PCF,REF_CFP,PCF_CFPS_LTM,CFP_CFPS_LTM,CFP_OpCF_MktCap,PCF_OpCF_MktCap
0,SRENH.S,2024-01-03,14.766154,0.067722,<NA>,<NA>,0.116647,8.572903
1,SRENH.S,2024-01-04,14.875927,0.067223,<NA>,<NA>,0.116019,8.61928
2,SRENH.S,2024-01-05,14.891835,0.067151,<NA>,<NA>,0.115875,8.630008
3,SRENH.S,2024-01-08,15.055774,0.06642,<NA>,<NA>,0.114696,8.718712
4,SRENH.S,2024-01-09,14.939596,0.066936,<NA>,<NA>,0.115398,8.665642
...,...,...,...,...,...,...,...,...
5049,ANTO.L,2025-12-23,17.947406,0.055718,<NA>,<NA>,0.066271,15.089468
5050,ANTO.L,2025-12-24,17.867784,0.055967,<NA>,<NA>,0.066477,15.0429
5051,ANTO.L,2025-12-29,17.803669,0.056168,<NA>,<NA>,0.066682,14.996513
5052,ANTO.L,2025-12-30,18.328658,0.054559,<NA>,<NA>,0.064629,15.472894


In [51]:
# Vertailu: median |ero %| REF_PCF:ään, per kaava ja per osake
calc_cols = ["CALC_MktCap_div_OpCF", "CALC_Price_div_OpCF_Shares",
             "CALC_Price_div_CFPS_Act", "CALC_Price_div_OpCFPS"]

print("=== Kokonaistaso ===")
for col in calc_cols:
    mask = REF_PCF.notna() & df[col].notna()
    if mask.sum() > 0:
        diff = ((df.loc[mask, col] - REF_PCF[mask]) / REF_PCF[mask] * 100).abs()
        print(f"  {col}: median {diff.median():.1f}%, mean {diff.mean():.1f}%, max {diff.max():.1f}%")

print("\n=== Per osake (median |ero %|) ===")
for stock in STOCK:
    sub = df[df.Instrument == stock]
    ref = sub["Price To Cash Flow Per Share (Daily Time Series Ratio)"]
    print(f"\n{stock}:")
    for col in calc_cols:
        mask = ref.notna() & sub[col].notna()
        if mask.sum() > 0:
            diff = ((sub.loc[mask, col] - ref[mask]) / ref[mask] * 100).abs().median()
            print(f"    {col}: {diff:.1f}%")
        else:
            print(f"    {col}: ei dataa")

=== Kokonaistaso ===
  CALC_MktCap_div_OpCF: median 14.9%, mean 20.5%, max 68.2%
  CALC_Price_div_OpCF_Shares: median 16.5%, mean 23.3%, max 67.3%
  CALC_Price_div_CFPS_Act: median 6.1%, mean 8.7%, max 33.0%
  CALC_Price_div_OpCFPS: median 7.1%, mean 15.9%, max 1369.0%

=== Per osake (median |ero %|) ===

SRENH.S:
    CALC_MktCap_div_OpCF: 3.5%
    CALC_Price_div_OpCF_Shares: 9.7%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 4.7%

AKRBP.OL:
    CALC_MktCap_div_OpCF: 8.9%
    CALC_Price_div_OpCF_Shares: 9.0%
    CALC_Price_div_CFPS_Act: 3.7%
    CALC_Price_div_OpCFPS: 3.5%

TLIT.MI:
    CALC_MktCap_div_OpCF: 41.2%
    CALC_Price_div_OpCF_Shares: 59.4%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 43.4%

AAL.L:
    CALC_MktCap_div_OpCF: 22.6%


    CALC_Price_div_OpCF_Shares: 21.9%
    CALC_Price_div_CFPS_Act: 9.0%
    CALC_Price_div_OpCFPS: 4.0%

ABVX.PA:
    CALC_MktCap_div_OpCF: ei dataa
    CALC_Price_div_OpCF_Shares: ei dataa
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: ei dataa

SVT.L:
    CALC_MktCap_div_OpCF: 38.7%
    CALC_Price_div_OpCF_Shares: 38.7%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 31.1%

FER.MC:
    CALC_MktCap_div_OpCF: 15.2%
    CALC_Price_div_OpCF_Shares: 11.0%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 18.6%

RAAG.DE:
    CALC_MktCap_div_OpCF: 8.0%
    CALC_Price_div_OpCF_Shares: 8.0%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 3.8%

EQTAB.ST:
    CALC_MktCap_div_OpCF: 11.5%
    CALC_Price_div_OpCF_Shares: 15.8%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCFPS: 4.0%

ANTO.L:
    CALC_MktCap_div_OpCF: 7.4%
    CALC_Price_div_OpCF_Shares: 7.4%
    CALC_Price_div_CFPS_Act: ei dataa
    CALC_Price_div_OpCF

In [52]:
# NaN-kattavuus
print("NaN %:")
for c in df.columns:
    pct = df[c].isna().mean() * 100
    if pct > 0:
        print(f"  {c}: {pct:.1f}%")

NaN %:
  Price To Cash Flow Per Share (Daily Time Series Ratio): 12.3%
  Cash Flow Per Share - Actual: 84.4%
  CALC_Price_div_CFPS_Act: 80.1%
  CFPS_LTM: 85.1%
  CALC_Price_div_CFPS_LTM: 85.1%
